Layer 1

In [32]:
# 0. Imports
import os
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm
import timm
import numpy as np
import pandas as pd
# Ensure scripts folder is in path
from scripts.prepare_data import prepare_data
from scripts.datasets import GeoguessrDataset
from scripts.model import GeoguessrModel
import importlib
from scripts.losses import CoordinateLoss, haversine_distance

In [11]:
# 1. SETUP AND DATA PREPARATION
print("Checking data preparation...")
prepare_data()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device} | RTX 5070 Optimizations Enabled")
os.makedirs("saved_models", exist_ok=True)

base_dir = os.path.abspath('.')
csv_path = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'enriched_training_data.csv')
image_dir = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'images')
geojson_path = os.path.join(base_dir, 'country_boundaries.geojson')

print("\nLoading dataset...")
dataset = GeoguessrDataset(
    csv_path=csv_path, image_dir=image_dir, transform=None, geojson_path=geojson_path
)
num_countries = dataset.get_num_classes()

print(f"\nDataset loaded successfully with {len(dataset)} images and {num_countries} unique countries.")
print("--- Top 6 Countries ---")
print(dataset.df['country_name'].value_counts().head(6))

Checking data preparation...
Loading coordinates from c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\ground_truth_coordinates.csv...
Creating GeoDataFrame...
Loading country boundaries from c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\country_boundaries.geojson...
Performing spatial join (this may take a minute)...
Found 2877 points outside strict boundaries (likely coasts/islands). Finding nearest country...
Final missing countries: 0
Saving enriched data to c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\enriched_training_data.csv...
Done! Data preparation is complete.
Using device: cuda | RTX 5070 Optimizations Enabled

Loading dataset...

Dataset loaded successfully with 19002 images and 258 unique countries.
--- Top 6 Countries ---
country_name
United States of America    2135
Russia                      1916
Brazil                      1599
Australia                   1457
Canada       

In [3]:
# 2. EVALUATION FUNCTION
def evaluate_model(model, dataset, batch_size=64, workers=8):
    print("\n--- Evaluating Model Performance ---")
    model.eval() # Freeze layers like Dropout/BatchNorm for testing
    
    # We use a standard DataLoader here to test every image exactly once!
    eval_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=workers)
    criterion = nn.CrossEntropyLoss()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(eval_loader, desc="Evaluating")
    
    with torch.no_grad(): # Disable calculus to save memory and run faster
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                country_logits = outputs['country_logits']
                loss = criterion(country_logits, labels)
            
            total_loss += loss.item()
            
            # Find the index of the highest probability
            predictions = torch.argmax(country_logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
    avg_loss = total_loss / len(eval_loader)
    accuracy = (correct / total) * 100
    
    print(f"--> Evaluation Complete!")
    print(f"--> Average Loss: {avg_loss:.4f}")
    print(f"--> Accuracy: {accuracy:.2f}% ({correct} / {total} correct)\n")

In [4]:
# 3. THE EXPERIMENT COMMAND CENTER
def run_experiment(backbone_name, batch_size=64, epochs=3, workers=8, learning_rate=1e-4):
    print(f"\n{'='*50}")
    print(f"STARTING EXPERIMENT: {backbone_name}")
    print(f"{'='*50}")
    
    # 1. Dynamically get the required image size and normalizations
    dummy_model = timm.create_model(backbone_name, pretrained=False)
    data_config = timm.data.resolve_data_config({}, model=dummy_model)
    required_image_size = data_config['input_size'][1] 
    print(f"The model '{backbone_name}' requires image size: {required_image_size}x{required_image_size}")
    # 2. Create the dynamic transform
    transform = A.Compose([
        A.Resize(required_image_size, required_image_size),
        A.Normalize(mean=data_config['mean'], std=data_config['std']),
        ToTensorV2()
    ])
    
    # 3. Inject the transform into the existing dataset
    dataset.transform = transform
    
    # 4. Initialize the actual model
    model = GeoguessrModel(num_countries=num_countries, backbone_name=backbone_name, pretrained=True)
    model = model.to(device)
    
    # --- RESUME CAPABILITY ---
    save_path = f"saved_models/layer1_{backbone_name}.pth"
    if os.path.exists(save_path):
        print(f"--> Found existing weights! Loading {save_path} to resume.")
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    else:
        print("--> No existing weights found. Starting fresh.")
        
    if epochs == 0:
        print("--> Epochs set to 0. Skipping training.")
        evaluate_model(model, dataset, batch_size)
        return model
    persist = True if workers > 0 else False
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=workers,
                             persistent_workers=persist)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs} [{backbone_name}]")
        
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs['country_logits'], labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        epoch_loss = running_loss / len(dataloader)
        print(f"--> Epoch {epoch+1} Average Training Loss: {epoch_loss:.4f}")
        torch.save(model.state_dict(), save_path)
        
    print(f"SUCCESS! Model safely saved to: {save_path}")
    
    # Run a final evaluation after all epochs are done!
    evaluate_model(model, dataset, batch_size)
    return model

In [ ]:
# 4. RUN YOUR EXPERIMENTS HERE
model_b0 = run_experiment(backbone_name='efficientnet_b0', batch_size=64, epochs=4)


STARTING EXPERIMENT: efficientnet_b0
The model 'efficientnet_b0' requires image size: 224x224


--> No existing weights found. Starting fresh.


Epoch 1/4 [efficientnet_b0]: 100%|██████████| 297/297 [03:42<00:00,  1.33it/s, loss=2.9337]  


--> Epoch 1 Average Training Loss: 3.6789


Epoch 2/4 [efficientnet_b0]: 100%|██████████| 297/297 [00:38<00:00,  7.67it/s, loss=2.5985]


...

--> Epoch 3 Average Training Loss: 2.2371


Epoch 4/4 [efficientnet_b0]: 100%|██████████| 297/297 [00:38<00:00,  7.79it/s, loss=1.6613]


--> Epoch 4 Average Training Loss: 1.7625
SUCCESS! Model safely saved to: saved_models/layer1_efficientnet_b0.pth

--- Evaluating Model Performance ---


Evaluating: 100%|██████████| 297/297 [01:03<00:00,  4.70it/s]


--> Evaluation Complete!
--> Average Loss: 1.2765
--> Accuracy: 68.53% (13022 / 19002 correct)



In [18]:
model_v2_small = run_experiment(backbone_name='efficientnetv2_rw_s', batch_size=40, epochs=0, learning_rate=6e-5)


STARTING EXPERIMENT: efficientnetv2_rw_s
The model 'efficientnetv2_rw_s' requires image size: 288x288
--> Found existing weights! Loading saved_models/layer1_efficientnetv2_rw_s.pth to resume.
--> Epochs set to 0. Skipping training.

--- Evaluating Model Performance ---


Evaluating: 100%|██████████| 476/476 [01:13<00:00,  6.51it/s]

--> Evaluation Complete!
--> Average Loss: 0.9569
--> Accuracy: 76.90% (14613 / 19002 correct)



PHASE 2: COORDINATE REGRESSOR TRAINING (STANDALONE)

In [20]:
# 5. Layer 2 training
def train_phase2_coordinates(model, dataset, backbone_name, epochs=5, batch_size=64, workers=8, lr=1e-4):
    print("\n" + "="*50)
    print(f"STARTING PHASE 2: COORDINATE REGRESSOR [{backbone_name}]")
    print("="*50)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # 1. Freeze Backbone and Country Head (Protect Layer 1's accuracy!)
    print("Freezing Vision Backbone and Layer 1...")
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.country_head.parameters():
        param.requires_grad = False
        
    # Ensure ONLY the Coordinate Head learns
    for param in model.coordinate_head.parameters():
        param.requires_grad = True
        
    # 2. Setup Dataloader and Optimizer
    persist = True if workers > 0 else False
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=workers,
                            persistent_workers=persist)
    
    criterion = CoordinateLoss()
    # Note: We only pass the coordinate_head parameters to the optimizer!
    optimizer = optim.AdamW(model.coordinate_head.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda')
    
    num_countries = model.country_head.out_features
    
    # Ensure saved_models directory exists
    os.makedirs("saved_models", exist_ok=True)
    save_path = f"saved_models/layer2_{backbone_name}.pth"
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Phase 2 - Epoch {epoch+1}/{epochs} [{backbone_name}]")
        
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            lats = batch['latitude'].to(device, non_blocking=True)
            lons = batch['longitude'].to(device, non_blocking=True)
            
            # --- TARGET SMOOTHING (80% True / 20% Others) ---
            # 1. Create a probability matrix initialized with the uniform remaining probability
            uniform_prob = 0.257 / (num_countries - 1)
            force_probs = torch.full((images.size(0), num_countries), uniform_prob, device=device)
            # 2. Inject the 80% probability at the correct ground-truth country indices
            force_probs.scatter_(1, labels.unsqueeze(1), 0.743)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                # Pass the images AND our mathematically perfect 80% country probabilities
                outputs = model(images, force_country_probs=force_probs)
                
                # Calculate Coordinate Loss using 3D Unit Sphere MSE
                loss = criterion(outputs['pred_xyz'], lats, lons)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'xyz_mse_loss': f"{loss.item():.5f}"})
            
        epoch_loss = running_loss / len(dataloader)
        print(f"--> Epoch {epoch+1} Average Training Loss: {epoch_loss:.5f}")
        
        # Save Phase 2 weights
        torch.save(model.state_dict(), save_path)
        
    print(f"SUCCESS! Phase 2 Model safely saved to: {save_path}")
    return model

In [21]:
# 6. EVAL FOR LAYER 1 + LAYER 2
def evaluate_phase2_coordinates(model, dataset, batch_size=64, workers=8):
    print("\n--- Evaluating Layer 2 (Coordinate Regressor) ---")
    
    device = next(model.parameters()).device
    model.eval() # Freeze layers like Dropout/BatchNorm for testing
    
    # Standard DataLoader to test every image exactly once
    eval_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=workers)
    
    all_distances_km = []
    
    progress_bar = tqdm(eval_loader, desc="Evaluating Distances")
    
    with torch.no_grad(): # Disable gradients to save memory and run faster
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            true_lats = batch['latitude'].to(device, non_blocking=True)
            true_lons = batch['longitude'].to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                # Note: We do NOT use force_country_probs here. 
                # We want to evaluate how it performs using Layer 1's actual predictions!
                outputs = model(images)
                pred_lats = outputs['pred_lat']
                pred_lons = outputs['pred_lon']
                
                # Calculate exact Haversine distance in km
                distances = haversine_distance(pred_lats, pred_lons, true_lats, true_lons)
                
            all_distances_km.extend(distances.cpu().numpy().tolist())
            
    # Calculate metrics
    mean_distance = np.mean(all_distances_km)
    median_distance = np.median(all_distances_km)
    
    print(f"--> Evaluation Complete!")
    print(f"--> Mean Distance Error: {mean_distance:.2f} km")
    print(f"--> Median Distance Error: {median_distance:.2f} km")
    print(f"--> (Note: The competition judges you on the MEDIAN score!)\n")

In [ ]:
# 7. RELOAD
import importlib
import scripts.model

# 1. Force Python to reload the updated model.py script
importlib.reload(scripts.model)
from scripts.model import GeoguessrModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_countries = dataset.get_num_classes()

# 2. Instantiate the NEW model (which now has the coordinate_head)
model_v2_small = GeoguessrModel(num_countries=num_countries, backbone_name='efficientnetv2_rw_s', pretrained=False)
model_v2_small = model_v2_small.to(device)

# 3. Load your Phase 1 weights back into it!
phase1_weights_path = "saved_models/layer1_efficientnetv2_rw_s.pth"
print(f"Loading Phase 1 weights from {phase1_weights_path}...")
model_v2_small.load_state_dict(torch.load(phase1_weights_path, map_location=device, weights_only=True), strict=False)
print("Weights loaded successfully!")

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x0000025E2FE70D60>
Traceback (most recent call last):
  File "c:\Users\Yash T\Desktop\geoguessr\.venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1686, in __del__
    self._shutdown_workers()
  File "c:\Users\Yash T\Desktop\geoguessr\.venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1645, in _shutdown_workers
    self._mark_worker_as_unavailable(worker_id, shutdown=True)
  File "c:\Users\Yash T\Desktop\geoguessr\.venv\Lib\site-packages\torch\utils\data\dataloader.py", line 1572, in _mark_worker_as_unavailable
    not self._workers_status[worker_id]
        ^^^^^^^^^^^^^^^^^^^^
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


Loading Phase 1 weights from saved_models/layer1_efficientnetv2_rw_s.pth...
Weights loaded successfully!


In [26]:
# 6. RUN YOUR EXPERIMENTS HERE
model_b0 = train_phase2_coordinates(model=model_v2_small, dataset=dataset, backbone_name='efficientnetv2_rw_s',
                                    epochs=4, batch_size=40, lr=6e-5)


STARTING PHASE 2: COORDINATE REGRESSOR [efficientnetv2_rw_s]
Freezing Vision Backbone and Layer 1...


Phase 2 - Epoch 1/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [01:03<00:00,  7.47it/s, xyz_mse_loss=0.98950]


--> Epoch 1 Average Training Loss: 0.10372


Phase 2 - Epoch 2/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [00:36<00:00, 13.10it/s, xyz_mse_loss=0.18848]


--> Epoch 2 Average Training Loss: 0.06864


Phase 2 - Epoch 3/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [00:36<00:00, 13.18it/s, xyz_mse_loss=0.73693]


--> Epoch 3 Average Training Loss: 0.05989


Phase 2 - Epoch 4/4 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [00:36<00:00, 13.15it/s, xyz_mse_loss=0.41563]


--> Epoch 4 Average Training Loss: 0.05054
SUCCESS! Phase 2 Model safely saved to: saved_models/layer2_efficientnetv2_rw_s.pth


In [31]:
evaluate_phase2_coordinates(model_v2_small, dataset, batch_size=40)


--- Evaluating Layer 2 (Coordinate Regressor) ---


Evaluating Distances: 100%|██████████| 476/476 [01:01<00:00,  7.79it/s]

--> Evaluation Complete!
--> Mean Distance Error: 1816.71 km
--> Median Distance Error: 1270.69 km
--> (Note: The competition judges you on the MEDIAN score!)



In [37]:
# 7. GENERATE 1ST SUBMISSION
from PIL import Image
def generate_submission(model, backbone_name, sample_sub_path, test_image_dir, output_path="submission.csv"):
    print("\n" + "="*50)
    print("STARTING TEST SET INFERENCE")
    print("="*50)
    
    device = next(model.parameters()).device
    model.eval()
    
    # 1. Load sample submission
    df = pd.read_csv(sample_sub_path)
    print(f"Loaded sample submission with {len(df)} images.")
    
    # 2. Re-create the exact transform used during training
    dummy_model = timm.create_model(backbone_name, pretrained=False)
    data_config = timm.data.resolve_data_config({}, model=dummy_model)
    required_image_size = data_config['input_size'][1] 
    
    transform = A.Compose([
        A.Resize(required_image_size, required_image_size),
        A.Normalize(mean=data_config['mean'], std=data_config['std']),
        ToTensorV2()
    ])
    
    predictions = []
    
    # 3. Iterate over the test images
    progress_bar = tqdm(df['image_id'], desc="Predicting Locations")
    
    with torch.no_grad(): # Disable gradients for massive speedup
        for image_id in progress_bar:
            img_path = os.path.join(test_image_dir, image_id)
            
            # Load and preprocess image
            try:
                image = Image.open(img_path).convert('RGB')
                image_np = np.array(image)
                augmented = transform(image=image_np)
                # Add batch dimension of 1 since we are processing one by one
                image_tensor = augmented['image'].unsqueeze(0).to(device) 
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                # Fallback to (0,0) if the image file is corrupt/missing
                predictions.append({
                    'image_id': image_id,
                    'pred_lat': 0.0,
                    'pred_lon': 0.0,
                    'pred_radius_km': 1500.0
                })
                continue
            
            # Inference!
            with torch.amp.autocast('cuda'):
                # Note: No force_country_probs here! True unassisted inference.
                outputs = model(image_tensor)
                pred_lat = outputs['pred_lat'].item()
                pred_lon = outputs['pred_lon'].item()
            
            # Append prediction (hardcoding radius to 2000 for Phase 2)
            predictions.append({
                'image_id': image_id,
                'pred_lat': pred_lat,
                'pred_lon': pred_lon,
                'pred_radius_km': 1500.0
            })
            
    # 4. Save to CSV in exactly the required format
    pred_df = pd.DataFrame(predictions)
    pred_df.to_csv(output_path, index=False)
    print(f"SUCCESS! Submission saved to: {os.path.abspath(output_path)}")

In [38]:
generate_submission(
    model=model_v2_small, backbone_name='efficientnetv2_rw_s',
    sample_sub_path='sample_submission.csv', test_image_dir='test_images_sampled', output_path='submission.csv'
)


STARTING TEST SET INFERENCE
Loaded sample submission with 500 images.


Predicting Locations: 100%|██████████| 500/500 [00:23<00:00, 21.54it/s]

SUCCESS! Submission saved to: c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\submission.csv
